# <center> Differential expression analysis

## <center> Description

- env: squidpy_v06

## <center> Set-up

#### **libs**

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import decoupler as dc
import sys
from pathlib import Path
from scipy import sparse
## for vst
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

## plotting
# import marsilea as ma
# import marsilea.plotter as mp
# from matplotlib.colors import NoNorm # for marsilea dotplot
import plotnine as p9
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

#### **paths**

In [2]:
from pathlib import Path

if "snakemake" in globals():
    # in_metadata = Path(snakemake.input["in_metadata"])
    # out_metadata = Path(snakemake.output["out_metadata"])
    print('Hello World')
else:
    ## get paths
    ROOT = get_project_root(globals())
    ## input
    in_adata = ROOT / "output/preprocessing/objects/QC/adata.h5ad"
    ## output
    ### objects
    out_deg = ROOT / "output/DEA/objects/pyDESeq2/deg.tsv"
    out_deg.parent.mkdir(parents=True, exist_ok=True) # make sure the output folder exists
    ### plots
    out_plots = ROOT / "output/DEA/plots/pyDESeq2"
    out_plots.mkdir(parents=True, exist_ok=True)


#### **reproducibility**

In [3]:
import random
import numpy as np
import torch
import scanpy.external as sce

SEED = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## <center> Main

#### **Load data**

In [4]:
adata = sc.read_h5ad(in_adata)

adata

AnnData object with n_obs × n_vars = 66 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [5]:
adata.obs

,disease_group,QC_group,condition_group,grp,leiden_0.05,leiden_0.1,leiden_0.15,leiden_0.2
sample_id,,,,,,,,
SD1,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0
SD3,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0
SD5,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0
SD7,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0
SD9,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0
...,...,...,...,...,...,...,...,...
SD58,HR_control,A,IFNG_pos,HR_control__IFNG_pos,0,0,0,0
SD60,HR_control,A,IFNG_pos,HR_control__IFNG_pos,0,0,0,0
SD62,HR_control,A,IFNG_pos,HR_control__IFNG_pos,0,0,0,2


In [8]:
col_dict = {
    'SD1': 'individual_01',
    'SD2': 'individual_01',
    'SD3': 'individual_02',
    'SD4': 'individual_02',
    'SD5': 'individual_03',
    'SD6': 'individual_03',
    'SD7': 'individual_04',
    'SD8': 'individual_04',
    'SD9': 'individual_05',
    'SD10': 'individual_05',
    'SD11': 'individual_06',
    'SD12': 'individual_06',
    'SD13': 'individual_07',
    'SD14': 'individual_07',
    'SD15': 'individual_08',
    'SD16': 'individual_08',
    'SD17': 'individual_09',
    'SD18': 'individual_09',
    'SD19': 'individual_10',
    'SD20': 'individual_10',
    'SD21': 'individual_11',
    'SD22': 'individual_11',
    'SD23': 'individual_12',
    'SD24': 'individual_12',
    'SD25': 'individual_13',
    'SD26': 'individual_13',
    'SD27': 'individual_14',
    'SD28': 'individual_14',
    'SD29': 'individual_15',
    'SD30': 'individual_15',
    'SD31': 'individual_16',
    'SD32': 'individual_16',
    'SD33': 'individual_17',
    'SD34': 'individual_17',
    'SD35': 'individual_18',
    'SD36': 'individual_18',
    'SD37': 'individual_19',
    'SD38': 'individual_19',
    'SD39': 'individual_20',
    'SD40': 'individual_20',
    'SD41': 'individual_21',
    'SD42': 'individual_21',
    'SD43': 'individual_22',
    'SD44': 'individual_22',
    'SD45': 'individual_23',
    'SD46': 'individual_23',
    'SD47': 'individual_24',
    'SD48': 'individual_24',
    'SD49': 'individual_25',
    'SD50': 'individual_25',
    'SD51': 'individual_26',
    'SD52': 'individual_26',
    'SD53': 'individual_27',
    'SD54': 'individual_27',
    'SD55': 'individual_28',
    'SD56': 'individual_28',
    'SD57': 'individual_29',
    'SD58': 'individual_29',
    'SD59': 'individual_30',
    'SD60': 'individual_30',
    'SD61': 'individual_31',
    'SD62': 'individual_31',
    'SD63': 'individual_32',
    'SD64': 'individual_32',
    'SD65': 'individual_33',
    'SD66': 'individual_33',
}

In [ ]:
# add individualID
adata.obs['individualID']=adata.obs.index.map(col_dict)


adata.obs.head()

,disease_group,QC_group,condition_group,grp,leiden_0.05,leiden_0.1,leiden_0.15,leiden_0.2,individualID
sample_id,,,,,,,,,
SD1,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0,individual_01
SD3,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0,individual_02
SD5,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0,individual_03
SD7,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0,individual_04
SD9,HR_case,A,IFNG_neg,HR_case__IFNG_neg,0,0,0,0,individual_05


In [10]:
# remove group B

adata = adata[adata.obs['QC_group']=='A'].copy()

adata

AnnData object with n_obs × n_vars = 50 × 19557
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2', 'individualID'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [11]:
# make sure counts are raw

adata.X = adata.layers['cts.raw'].copy()

In [12]:
# check dist of samples across groups

adata.obs[['disease_group', 'condition_group']].value_counts()

disease_group  condition_group
HR_control     IFNG_neg           13
               IFNG_pos           13
HR_case        IFNG_neg           12
               IFNG_pos           12
Name: count, dtype: int64

In [13]:
# create the combined DEA group used for the simpler one-factor design

# These are the only levels used in this 2 x 2 test analysis.
condition_levels = ["IFNG_neg", "IFNG_pos"]
disease_levels = ["HR_control", "HR_case"]

# Keep only samples belonging to the requested 2 x 2 design.
adata = adata[
    adata.obs["condition_group"].isin(condition_levels)
    & adata.obs["disease_group"].isin(disease_levels)
].copy()

# Store the original factors as categorical for clear ordering in summaries.
adata.obs["condition_group"] = pd.Categorical(
    adata.obs["condition_group"],
    categories=condition_levels,
)
adata.obs["disease_group"] = pd.Categorical(
    adata.obs["disease_group"],
    categories=disease_levels,
)

# Merge disease_group and condition_group into the requested combined factor.
# The separator is double underscore and the disease group comes first, e.g. HR_case__IFNG_pos.
DEA_group_levels = [
    "HR_control__IFNG_pos",
    "HR_case__IFNG_pos",
    "HR_control__IFNG_neg",
    "HR_case__IFNG_neg",
]
adata.obs["DEA_group"] = pd.Categorical(
    adata.obs["disease_group"].astype(str) + "__" + adata.obs["condition_group"].astype(str),
    categories=DEA_group_levels,
)

# Sanity checks before model fitting.
assert adata.n_obs > 0, "No samples left after subsetting to the requested DEA groups."
assert not adata.obs[["condition_group", "disease_group", "DEA_group"]].isna().any().any(), "Unexpected factor levels found."

dea_sample_counts = (
    adata.obs[["DEA_group", "disease_group", "condition_group"]]
    .value_counts(sort=False)
    .rename("n_samples")
    .reset_index()
)

# A one-factor model with all combined levels needs samples in every level kept here.
assert (dea_sample_counts["n_samples"] > 0).all(), "The combined DEA_group design has an empty group."

dea_sample_counts


,DEA_group,disease_group,condition_group,n_samples
0,HR_control__IFNG_pos,HR_control,IFNG_pos,13
1,HR_case__IFNG_pos,HR_case,IFNG_pos,12
2,HR_control__IFNG_neg,HR_control,IFNG_neg,13
3,HR_case__IFNG_neg,HR_case,IFNG_neg,12


## <center> DEA

In [14]:
# prepare data for pyDESeq2

# Use the raw count layer prepared in QC.ipynb.
# Genes were already filtered there, so no additional expression filtering is applied here.
adata_dds = adata.copy()
adata_dds.X = adata_dds.layers["cts.raw"].copy()

# pyDESeq2 expects an in-memory dense integer count matrix.
# The rounding is a safety step in case the count layer was stored as floating point.
if sparse.issparse(adata_dds.X):
    adata_dds.X = adata_dds.X.toarray()
adata_dds.X = np.rint(np.asarray(adata_dds.X)).astype(int)

# Keep all QC-filtered genes, but remove any accidental all-zero genes because DESeq2 cannot estimate them.
nonzero_genes = np.asarray(adata_dds.X.sum(axis=0)).ravel() > 0
print(f"Genes before all-zero safety check: {adata_dds.n_vars}")
print(f"Genes after all-zero safety check: {nonzero_genes.sum()}")
adata_dds = adata_dds[:, nonzero_genes].copy()

adata_dds


Genes before all-zero safety check: 19557
Genes after all-zero safety check: 19556


AnnData object with n_obs × n_vars = 50 × 19556
    obs: 'disease_group', 'QC_group', 'condition_group', 'grp', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.2', 'individualID', 'DEA_group'
    var: 'genesymbol', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'QC_group_colors', 'condition_group_colors', 'disease_group_colors', 'grp_colors', 'hvg', 'leiden_0.05', 'leiden_0.1', 'leiden_0.15', 'leiden_0.15_colors', 'leiden_0.2', 'log1p', 'neighbors', 'pca', 'rank_obsm', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'cts.deseq2_norm', 'cts.deseq2_norm_log', 'cts.raw', 'cts.vst'
    obsp: 'connectivities', 'distances'

In [ ]:
# fit the DESeq2 model with the simpler one-factor design

# DEA_group combines disease_group and condition_group.
# The test/reference direction is set explicitly in DeseqStats below.
#design_formula = "~ individualID + DEA_group"
design_formula = "~ DEA_group"

dds = DeseqDataSet(
    adata=adata_dds,
    design=design_formula,
    refit_cooks=True,
    n_cpus=1,  # kept at 1 because this environment has shown pyDESeq2 multiprocessing issues
)

# Run the standard DESeq2 workflow: size factors, dispersions, LFCs, and Cook's refitting if needed.
dds.deseq2()

# Check the design matrix columns so the contrast below is easy to audit.
dds.obsm["design_matrix"].head()


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/dds.py:308: UserWarning: The design matrix is not full rank, so the model cannot be fitted, but some operations like design-free VST remain possible. To perform differential expression analysis, please remove the design variables that are linear combinations of others.
Fitting size factors...
... done in 0.05 seconds.

/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:356: RuntimeWarning: divide by zero encountered in log


Using None as control genes, passed at DeseqDataSet initialization


/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:371: LinAlgWarning: Ill-conditioned matrix (rcond=1.10255e-16): result may not be accurate.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:371: LinAlgWarning: Ill-conditioned matrix (rcond=8.39791e-17): result may not be accurate.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:371: LinAlgWarning: Ill-conditioned matrix (rcond=8.60434e-17): result may not be accurate.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:371: LinAlgWarning: Ill-conditioned matrix (rcond=8.57719e-17): result may not be accurate.
/home/hd/hd_hd/hd_gb249/.conda/envs/mambaEnv/envs/squidpy_v06/lib/python3.10/site-packages/pydeseq2/utils.py:371: LinAlgWarning: Ill-conditioned matrix (rcond=9.36169e-17): result may not be accur

LinAlgError: Singular matrix

In [ ]:
# extract the requested DEA_group contrast

# This compares HR_case__IFNG_pos against HR_control__IFNG_pos.
# Therefore, a positive log2FC means higher expression in HR_case__IFNG_pos.
stat_res = DeseqStats(
    dds,
    contrast=["DEA_group", "HR_case__IFNG_pos", "HR_control__IFNG_pos"],
    alpha=0.05,
    cooks_filter=True,
    independent_filter=True,
    n_cpus=1,
)
stat_res.summary()

# Format results and add gene symbols when they are available in adata.var.
deg = stat_res.results_df.copy()
deg.index.name = "gene_id"

if "genesymbol" in adata_dds.var.columns:
    deg.insert(0, "genesymbol", adata_dds.var["genesymbol"].reindex(deg.index).astype(str))

deg["contrast"] = "HR_case__IFNG_pos_vs_HR_control__IFNG_pos"
deg["significant_pvalue_0.05"] = deg["pvalue"] < 0.05
deg["significant_padj_0.05"] = deg["padj"] < 0.05
deg["direction"] = np.select(
    [
        deg["log2FoldChange"] > 0,
        deg["log2FoldChange"] < 0,
    ],
    ["higher_HR_case__IFNG_pos", "higher_HR_control__IFNG_pos"],
    default="no_change",
)

# Keep the most significant genes at the top of the saved table.
deg = deg.sort_values(["padj", "pvalue"], na_position="last")

deg.head(20)


In [ ]:
# QC plot: P value histogram

# A broadly well-calibrated test often has a roughly uniform null component plus
# enrichment near zero when there is signal. Strong deviations can indicate model or batch issues.
plot_tab = deg.dropna(subset=["pvalue"]).copy()

w, h = 5, 4
p = (
    p9.ggplot(plot_tab, p9.aes(x="pvalue"))
    + p9.geom_histogram(
        breaks=np.arange(0, 1.05, 0.05),
        fill="#4C78A8",
        color="white",
        boundary=0,
    )
    + p9.scale_x_continuous(breaks=np.arange(0, 1.05, 0.1), limits=(0, 1))
    + p9.theme_classic()
    + p9.labs(
        title="P value distribution",
        x="Raw P value",
        y="Number of genes",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text=p9.element_text(size=10),
        figure_size=(w, h),
    )
)

p.save(out_plots / "pvalue_histogram.png", width=w, height=h, limitsize=False)
p.save(out_plots / "pvalue_histogram.pdf", width=w, height=h, limitsize=False)

plt.close()


In [ ]:
# QC plot: volcano plot with decoupler

# decoupler labels the top significant genes using the dataframe index.
# The y-axis uses adjusted P values so the significance threshold is FDR < 0.05.
# Use gene symbols for labels when available; otherwise fall back to gene IDs.
volcano_tab = deg.dropna(subset=["log2FoldChange", "padj"]).copy()
if "genesymbol" in volcano_tab.columns:
    volcano_tab["plot_label"] = volcano_tab["genesymbol"].where(
        volcano_tab["genesymbol"].notna() & (volcano_tab["genesymbol"] != "nan"),
        volcano_tab.index.astype(str),
    )
else:
    volcano_tab["plot_label"] = volcano_tab.index.astype(str)
volcano_tab = volcano_tab.set_index("plot_label", drop=False)

fig = dc.pl.volcano(
    data=volcano_tab,
    x="log2FoldChange",
    y="padj",
    top=20,
    thr_stat=0.5,
    thr_sign=0.05,
    figsize=(7, 6),
    dpi=120,
    return_fig=True,
)
fig.axes[0].set_title("HR_case__IFNG_pos vs HR_control__IFNG_pos")
fig.savefig(out_plots / "volcano.png", bbox_inches="tight")
fig.savefig(out_plots / "volcano.pdf", bbox_inches="tight")

plt.close(fig)


In [ ]:
# QC plot: number of genes passing raw and adjusted P value thresholds

# Count genes passing the two requested significance thresholds.
sig_counts = pd.DataFrame(
    {
        "threshold": ["Raw P < 0.05", "Padj < 0.05"],
        "n_genes": [
            int((deg["pvalue"] < 0.05).sum()),
            int((deg["padj"] < 0.05).sum()),
        ],
    }
)

w, h = 4.5, 4
p = (
    p9.ggplot(sig_counts, p9.aes(x="threshold", y="n_genes", fill="threshold"))
    + p9.geom_col(color="black", width=0.7, show_legend=False)
    + p9.geom_text(p9.aes(label="n_genes"), va="bottom", size=10, format_string="{:.0f}")
    + p9.scale_fill_manual(["#4C78A8", "#F58518"])
    + p9.theme_classic()
    + p9.labs(
        title="Significant genes",
        x="",
        y="Number of genes",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text_x=p9.element_text(size=10, rotation=20, ha="right"),
        axis_text_y=p9.element_text(size=10),
        figure_size=(w, h),
    )
)

p.save(out_plots / "significant_gene_counts.png", width=w, height=h, limitsize=False)
p.save(out_plots / "significant_gene_counts.pdf", width=w, height=h, limitsize=False)

plt.close()

sig_counts


In [ ]:
# QC plot: MA plot

# The MA plot shows the relationship between average normalized expression and log2FC.
# It is useful for spotting low-count instability or global shifts.
ma_tab = deg.dropna(subset=["baseMean", "log2FoldChange"]).copy()
ma_tab["is_padj_0.05"] = ma_tab["padj"] < 0.05

w, h = 5.5, 4.5
p = (
    p9.ggplot(ma_tab, p9.aes(x="baseMean", y="log2FoldChange"))
    + p9.geom_point(p9.aes(color="is_padj_0.05"), alpha=0.55, size=0.8)
    + p9.scale_x_log10()
    + p9.scale_color_manual({False: "#7F7F7F", True: "#D62728"})
    + p9.geom_hline(yintercept=0, linetype="dashed")
    + p9.theme_classic()
    + p9.labs(
        title="MA plot",
        x="Mean normalized counts",
        y="log2FC",
        color="Padj < 0.05",
    )
    + p9.theme(
        plot_title=p9.element_text(hjust=0.5, size=13),
        axis_title=p9.element_text(size=12),
        axis_text=p9.element_text(size=10),
        legend_title=p9.element_text(size=10),
        legend_text=p9.element_text(size=9),
        figure_size=(w, h),
    )
)

p.save(out_plots / "ma_plot.png", width=w, height=h, limitsize=False)
p.save(out_plots / "ma_plot.pdf", width=w, height=h, limitsize=False)

plt.close()


#### **Save results**

In [ ]:
deg.to_csv(out_deg,sep='\t')